# C3-gradient-descent — Practice p13 — Solution

The loss surface is a tilted elliptical bowl whose grid minimum is near $(w,b)=(1.8,0.55)$. With $\eta=0.2$, descent reaches $(1.79936,0.54207)$ and a loss of $0.0736257$, slightly below the grid minimum $0.0737018$ because it is not restricted to grid points. On the same surface $\eta=1.2$ overshoots and amplifies the error, producing a final loss above $10^{56}$.

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)
x = rng.uniform(0, 2, 80)
y = 1.8 * x + 0.6 + rng.normal(0, 0.25, 80)
w_vals = np.linspace(0, 3, 61)
b_vals = np.linspace(-1, 2, 61)

grid_residuals = b_vals[:, None, None] + w_vals[None, :, None] * x[None, None, :] - y[None, None, :]
L_grid = (grid_residuals ** 2).mean(axis=2)
grid_index = L_grid.argmin()
best_w_grid = w_vals[grid_index % 61]
best_b_grid = b_vals[grid_index // 61]
min_grid = L_grid.min()

def run_descent(eta, steps):
    w = 0.0
    b = 0.0
    for _ in range(steps):
        residuals = w * x + b - y
        grad_w = 2 * (residuals * x).mean()
        grad_b = 2 * residuals.mean()
        w = w - eta * grad_w
        b = b - eta * grad_b
    loss = ((w * x + b - y) ** 2).mean()
    return w, b, loss

w_fit, b_fit, final_loss = run_descent(0.2, 100)
grid_gap = max(abs(w_fit - best_w_grid), abs(b_fit - best_b_grid))
w_big, b_big, loss_big = run_descent(1.2, 40)
diverged = bool(loss_big > 1e6)
best_w_grid, best_b_grid, min_grid, w_fit, b_fit, final_loss, grid_gap, loss_big, diverged

### Answer check

In [ ]:
assert L_grid.shape == (61, 61)
assert np.allclose([best_w_grid, best_b_grid], [1.8, 0.55], atol=1e-12, rtol=0)
assert np.isclose(min_grid, 0.0737018262785, atol=1e-12, rtol=0)
assert np.allclose([w_fit, b_fit], [1.799362766855, 0.542072652317], atol=1e-12, rtol=0)
assert np.isclose(final_loss, 0.0736256937820, atol=1e-12, rtol=0)
assert np.isclose(grid_gap, 0.007927347683, atol=1e-12, rtol=0) and final_loss < min_grid
assert np.isclose(loss_big, 1.849443819681027e56, atol=1e42, rtol=0) and diverged is True